In [26]:
import cv2
import numpy as np
import joblib
import os
import urllib.request
from collections import deque
from mediapipe import Image, ImageFormat
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [27]:
model = joblib.load('posture_model_optuna.pkl')
scaler = joblib.load('scaler_optuna.pkl')

In [28]:
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = "pose_landmarker_lite.task"
if not os.path.exists(MODEL_PATH):
    print("📥 Downloading pose model (~10 MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✅ Downloaded")


In [29]:
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
detector = vision.PoseLandmarker.create_from_options(options)

In [30]:
LANDMARK_INDICES = list(range(15)) 

def compute_features(landmarks):
    pts = np.array([[landmarks[i].x, landmarks[i].y, landmarks[i].z] for i in LANDMARK_INDICES])
    nose = pts[0]
    left_shoulder = pts[11]; right_shoulder = pts[12]
    left_elbow = pts[13]; right_elbow = pts[14]

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
    if shoulder_width < 1e-6:
        shoulder_width = 1.0

    f = []
    vec = nose - shoulder_mid
    vertical = np.array([0, 1, 0])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    shoulder_vec = right_shoulder - left_shoulder
    horizontal = np.array([1, 0, 0])
    cos_slope = np.dot(shoulder_vec, horizontal) / (np.linalg.norm(shoulder_vec) * np.linalg.norm(horizontal) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_slope, -1.0, 1.0))))

    f.append(np.linalg.norm(nose - shoulder_mid) / shoulder_width)

    f.append((left_elbow[2] - left_shoulder[2]) / shoulder_width)
    f.append((left_elbow[0] - left_shoulder[0]) / shoulder_width)
    f.append((left_elbow[1] - left_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(left_elbow - left_shoulder) / shoulder_width)

    f.append((right_elbow[2] - right_shoulder[2]) / shoulder_width)
    f.append((right_elbow[0] - right_shoulder[0]) / shoulder_width)
    f.append((right_elbow[1] - right_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(right_elbow - right_shoulder) / shoulder_width)

    f.append((left_shoulder[1] - right_shoulder[1]) / shoulder_width)

    vec_proj = vec.copy(); vec_proj[1] = 0
    shoulder_proj = shoulder_vec.copy(); shoulder_proj[1] = 0
    if np.linalg.norm(vec_proj) > 0 and np.linalg.norm(shoulder_proj) > 0:
        cos_rot = np.dot(vec_proj, shoulder_proj) / (np.linalg.norm(vec_proj) * np.linalg.norm(shoulder_proj) + 1e-8)
        f.append(np.degrees(np.arccos(np.clip(cos_rot, -1.0, 1.0))))
    else:
        f.append(0.0)

    f.append(np.linalg.norm(left_elbow - right_elbow) / shoulder_width)

    return np.array(f).reshape(1, -1)

In [31]:
CONNECTIONS = [
    (0,1), (0,4), (1,2), (2,3), (4,5), (5,6), (7,0), (8,0),
    (9,0), (10,0), (11,12), (11,13), (12,14)
]

cap = cv2.VideoCapture(0)
print("Press 'q' to quit. Adjust threshold in code if needed.")

BUFFER_SECONDS = 0.5         
FPS_ESTIMATE = 30   
BUFFER_SIZE = int(BUFFER_SECONDS * FPS_ESTIMATE) 

prob_buffer = deque(maxlen=BUFFER_SIZE)

THRESHOLD = 0.3   

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = Image(image_format=ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_img)

    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]
        feat = compute_features(landmarks)
        feat_scaled = scaler.transform(feat)

        prob_good = model.predict_proba(feat_scaled)[0][0] 

        prob_buffer.append(prob_good)

        smoothed_prob = np.mean(prob_buffer) if prob_buffer else prob_good

        pred = 0 if smoothed_prob > THRESHOLD else 1
        label = "Good" if pred == 0 else "Bad"

        cv2.putText(frame, f"Posture: {label}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0) if pred == 0 else (0, 0, 255), 2)
        cv2.putText(frame, f"Smoothed prob: {smoothed_prob:.2f}  (inst: {prob_good:.2f})", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        h, w, _ = frame.shape
        for idx in LANDMARK_INDICES:
            lm = landmarks[idx]
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
        for (i, j) in CONNECTIONS:
            lm1, lm2 = landmarks[i], landmarks[j]
            x1, y1 = int(lm1.x * w), int(lm1.y * h)
            x2, y2 = int(lm2.x * w), int(lm2.y * h)
            cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    else:
        cv2.putText(frame, "No pose detected", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow('Posture Check', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()

Press 'q' to quit. Adjust threshold in code if needed.


In [ ]:



# ------------------------------------------------------------
# 2. Download MediaPipe pose model (if not present)
# ------------------------------------------------------------

# ------------------------------------------------------------
# 3. Initialise MediaPipe PoseLandmarker
# ------------------------------------------------------------


# ------------------------------------------------------------
# 4. Feature extraction (14 features, no wrists)
# ------------------------------------------------------------
LANDMARK_INDICES = list(range(15))   # only up to elbows (0..14)

def compute_features(landmarks):
    pts = np.array([[landmarks[i].x, landmarks[i].y, landmarks[i].z] for i in LANDMARK_INDICES])
    nose = pts[0]
    left_shoulder = pts[11]; right_shoulder = pts[12]
    left_elbow = pts[13]; right_elbow = pts[14]

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
    if shoulder_width < 1e-6:
        shoulder_width = 1.0

    f = []
    # 1. Head tilt
    vec = nose - shoulder_mid
    vertical = np.array([0, 1, 0])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    # 2. Shoulder slope
    shoulder_vec = right_shoulder - left_shoulder
    horizontal = np.array([1, 0, 0])
    cos_slope = np.dot(shoulder_vec, horizontal) / (np.linalg.norm(shoulder_vec) * np.linalg.norm(horizontal) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_slope, -1.0, 1.0))))

    # 3. Nose distance
    f.append(np.linalg.norm(nose - shoulder_mid) / shoulder_width)

    # 4-7. Left elbow relative
    f.append((left_elbow[2] - left_shoulder[2]) / shoulder_width)
    f.append((left_elbow[0] - left_shoulder[0]) / shoulder_width)
    f.append((left_elbow[1] - left_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(left_elbow - left_shoulder) / shoulder_width)

    # 8-11. Right elbow relative
    f.append((right_elbow[2] - right_shoulder[2]) / shoulder_width)
    f.append((right_elbow[0] - right_shoulder[0]) / shoulder_width)
    f.append((right_elbow[1] - right_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(right_elbow - right_shoulder) / shoulder_width)

    # 12. Shoulder asymmetry
    f.append((left_shoulder[1] - right_shoulder[1]) / shoulder_width)

    # 13. Head rotation
    vec_proj = vec.copy(); vec_proj[1] = 0
    shoulder_proj = shoulder_vec.copy(); shoulder_proj[1] = 0
    if np.linalg.norm(vec_proj) > 0 and np.linalg.norm(shoulder_proj) > 0:
        cos_rot = np.dot(vec_proj, shoulder_proj) / (np.linalg.norm(vec_proj) * np.linalg.norm(shoulder_proj) + 1e-8)
        f.append(np.degrees(np.arccos(np.clip(cos_rot, -1.0, 1.0))))
    else:
        f.append(0.0)

    # 14. Elbow-elbow distance
    f.append(np.linalg.norm(left_elbow - right_elbow) / shoulder_width)

    return np.array(f).reshape(1, -1)

# ------------------------------------------------------------
# 5. Skeleton connections (for visualisation)
# ------------------------------------------------------------
